<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_profiled_hash_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Profiled Hash Search

A fast, exact, gradient-free way to train the discrete **Ball Trap Brain**. No continuous relaxation, no genetic algorithm.

## The observation that makes this work

`ricochet` never reads the input. The input is consumed entirely by `travel`, which turns each token into a **start node**; after that, the ball's walk is a fixed function of that node (the deflector graph, hole symbols and energy rule never change with the input). So, exactly:

```
output  =  concat_i  T[ h_i ]

h_i = ( s[a_i] + Σ_{j≠i} j·d[a_j] )  mod D        <- a position-weighted hash (parameters: d, s)
T[k] = everything the ball emits when walked from node k      <- a table of D strings (parameters: the deflectors)
```

A ball can emit **as many symbols as it likes** (a chain of holes), so `T[k]` is a string of any length (up to `cap`), not a single symbol.

## Training = two halves, each solved exactly

1. **Table `T` — closed form.** For a fixed hash, the best table is the empirical majority string per bucket (or, with `loss='ce'`, the empirical class distribution — this is *variable projection*: the table is eliminated from the problem).
2. **Hash `(d, s)` — exact coordinate search.** For one token, remove the positions it affects from the bucket counts, then score **every** candidate value (all `2·direction_range+1` values of `d[a]`, or all `D` values of `s[a]`) in one vectorised gather, and verify the best ones exactly. This is hash-and-displace (CHD/PTHash-style perfect hashing), with "no collision" relaxed to "no *class* conflict".
3. **Latent split `target → one piece per ball`.** Since balls may emit several symbols, *how the target string is divided among the balls* is a free choice. We start from a canonical split and then re-split each example by dynamic programming so that the pieces agree with what the table already emits (hard EM; the loss never increases).
4. **Compile** the table into real deflectors: a bucket whose string is `"abc"` becomes a chain of hole deflectors `a → b → c → sink`, placed on nearest free nodes. We report the ball energy this needs.

Cost per sweep is `O(N·L²·candidates)` gathers and `O(D·C)` memory (C = number of distinct emitted strings). No autograd, no `D×D` matrices.

In [ ]:
import random, time, math, difflib
import numpy as np

CFG = dict(
    D=10000,                # number of deflectors (hash buckets)
    direction_range=10,  # d[a] in [-direction_range, direction_range]
    bounce=10,             # initial ball energy = 2*bounce (compile() raises it if a chain needs more)
    cap=10,                # max deflector hits per ball  ==  max symbols one ball can emit
    n_cases=1000, train_share=0.8, seed=0,
    loss='ce',             # 'hard' = 0/1 class conflicts, 'ce' = conditional-entropy profile (keeps the soft information)
)
random.seed(CFG['seed']); np.random.seed(CFG['seed'])

## Data: tokenizer, alphabet, next-token test cases
Same corpus, tokenizer and task as before (input = last 5 tokens, target = next token). Two clean-ups: the alphabet is de-duplicated (the old one had 24,546 entries but only 6,042 unique — duplicates were dead parameters), and the output alphabet is read off the targets instead of a hand-written list, so no target character is unreachable.

In [ ]:
def TokenizeInput(input, alphabet):
    """Greedy non-overlapping tokenization: earliest start, then longest, then alphabet order."""
    input_str = str(input)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda m: (m[0], m[1], m[2]))
    out, last_end = [], 0
    for index, neg_len, _, item in matches:
        if index >= last_end:
            out.append(item)
            last_end = index - neg_len
    return out

def substrings(s, max_len=5):
    seen = {}
    for i in range(len(s)):
        for j in range(i + 1, min(i + max_len, len(s)) + 1):
            seen.setdefault(s[i:j], None)
    return list(seen)

def make_alphabet(text, max_len=5):
    seen = {}
    for word in dict.fromkeys(text.split(' ')):
        for piece in substrings(word, max_len):
            seen.setdefault(piece, None)
    return list(seen)

def next_token_cases(text, amount, window=5, max_len=5):
    X = TokenizeInput(text, substrings(text, max_len))
    return [(''.join(X[max(0, i - window + 1):i + 1]), X[i + 1]) for i in range(amount)]

alphabet = [f'{i}' for i in range(10)] + list('=+/ ')
cases = [(f'1+{i}=', f'{i+1}') for i in range(CFG['n_cases'])]
random.shuffle(cases)
n_train = int(CFG['train_share'] * len(cases))
train_cases, test_cases = cases[:n_train], cases[n_train:]
output = sorted(set(''.join(t for _, t in train_cases)))
print(f"alphabet={len(alphabet)}  output symbols={len(output)}  train={len(train_cases)}  test={len(test_cases)}")
print(train_cases[:3])

alphabet=14  output symbols=10  train=800  test=200
[('1+439=', '440'), ('1+621=', '622'), ('1+160=', '161')]


## The discrete brain
Same machine as the original (`travel` → start deflector → ricochet walk with the energy rule), reduced to what is needed to run it. A hole emits its symbol, so a ball that hits several holes emits several symbols. `build_table()` exposes the decomposition above.

In [ ]:
class BallTrapBrain:
    def __init__(self, alphabet, output, D=7000, bounce=10, cap=10, direction_range=1000):
        self.alphabet = list(alphabet)
        self.index = {}
        for i, t in enumerate(self.alphabet):          # first occurrence, same as list.index()
            self.index.setdefault(t, i)
        self.output = list(output)
        self.D, self.cap, self.direction_range = D, cap, direction_range
        self.ball_energy = bounce * 2
        self.alphabet_directions = [random.randint(-direction_range, direction_range) for _ in self.alphabet]
        self.alphabet_start_positions = [random.randint(0, D - 1) for _ in self.alphabet]
        # deflector = [destination, value]; value is an int (bounce energy) or a str (hole symbol)
        self.deflectors_list = [[random.randint(0, D - 1), bounce] for _ in range(D)]

    def travel(self, directions, position, index):
        total = 0
        for j, d in enumerate(directions):
            if j != index:
                total += d * (j + 1)                   # position-weighted, leave-one-out
        return position + total

    def start_nodes(self, input):
        toks = [t for t in TokenizeInput(input, self.alphabet) if t in self.index]
        dirs = [self.alphabet_directions[self.index[t]] for t in toks]
        pos = [self.alphabet_start_positions[self.index[t]] for t in toks]
        return [self.travel(dirs, pos[i], i) % self.D for i in range(len(toks))]

    def walk(self, node):
        """Everything one ball emits when it starts on `node`."""
        out, dl, be, steps = [], self.deflectors_list, self.ball_energy, 0
        while be > 0 and steps < self.cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                out.append(val)
                be += 1 - dist
            node, steps = dest, steps + 1
        return ''.join(out)

    def compute_input(self, input):
        return ''.join(self.walk(k) for k in self.start_nodes(input))

    def build_table(self):
        """T[k]: the string emitted from each start node. compute_input == concat of T[h_i]."""
        return [self.walk(k) for k in range(self.D)]

    def display_deflectors(self, n=20):
        for i, defl in enumerate(self.deflectors_list[:n]):
            print(f"Deflector {i}: {defl}")


def evaluate(brain, cases):
    """(exact-match rate, mean difflib similarity) of brain.compute_input over (input, answer) cases."""
    exact = ratio = 0.0
    for inp, ans in cases:
        pred = brain.compute_input(inp)
        exact += pred == ans
        ratio += difflib.SequenceMatcher(None, pred, ans).ratio()
    return exact / len(cases), ratio / len(cases)

## The trainer

**Positions.** Every (example, ball) pair is one *position* `p` with a bucket `h_p`, and a *class* `c_p` = the string that ball should emit. The profiled loss of a hash is

* `hard`: `Σ_buckets (n_k − max_c N_kc)` — positions that disagree with their bucket's majority string;
* `ce`: `Σ_buckets [ n_k log n_k − Σ_c N_kc log N_kc ]` — the conditional entropy `N·H(class | bucket)`, i.e. the exact cross-entropy of the *best possible table* for this hash.

**Exact coordinate move.** Changing `d[a]` shifts position `p` (every ball *other than* the occurrence of `a`) by `w_p·Δ`, where `w_p` is the sum of the positions at which `a` occurs; changing `s[a]` shifts the balls that sit on `a` by `Δ`. For all candidate values `v` at once, the marginal loss of putting `p` in bucket `(base_p + w_p·v) mod D` is read straight off the count table, then the best few candidates are verified exactly and accepted only if the local loss does not increase.

In [ ]:
class HashTrainer:
    def __init__(self, brain, cases, loss='ce', resegment=True, max_len=None, seed=0):
        self.brain, self.loss, self.do_reseg = brain, loss, resegment
        self.D, self.V = brain.D, brain.direction_range
        self.max_len = min(max_len or brain.cap, brain.cap)
        self.rng = np.random.default_rng(seed)

        # ---- examples -> token ids + target strings
        self.ex_tokens, self.ex_target, dropped = [], [], 0
        out_set = set(brain.output)
        for inp, tgt in cases:
            toks = [brain.index[t] for t in TokenizeInput(inp, brain.alphabet)]
            if not toks or len(tgt) > len(toks) * self.max_len or not set(tgt) <= out_set:
                dropped += 1
                continue
            self.ex_tokens.append(toks); self.ex_target.append(tgt)
        # Contexts that tokenize identically get identical addresses for every ball, so no hash can give them
        # different answers. Keep the most common target per context and drop the rest (the exact-match ceiling).
        by_ctx = {}
        for toks, tgt in zip(self.ex_tokens, self.ex_target):
            by_ctx.setdefault(tuple(toks), []).append(tgt)
        keep = {k: max(set(v), key=lambda t: (v.count(t), -v.index(t))) for k, v in by_ctx.items()}
        pairs = [(k, t) for k, t in zip(self.ex_tokens, self.ex_target) if keep[tuple(k)] == t]
        ambiguous = len(self.ex_tokens) - len(pairs)
        self.ex_tokens, self.ex_target = [k for k, _ in pairs], [t for _, t in pairs]
        self.n_ex = len(self.ex_tokens)
        if dropped: print(f"[trainer] dropped {dropped} examples the brain cannot express")
        if ambiguous: print(f"[trainer] dropped {ambiguous} examples whose context is identical to another with a different target (unfittable)")

        # ---- classes = strings a single ball may emit ("" = emit nothing)
        self.cls_id, self.cls_str = {}, []
        self._reg("")
        for ch in brain.output: self._reg(ch)
        pieces = [self.canonical_split(t, len(k)) for k, t in zip(self.ex_tokens, self.ex_target)]
        for ps in pieces:
            for p in ps: self._reg(p)
        self.C = len(self.cls_str)

        # ---- flat position arrays
        lens = np.array([len(k) for k in self.ex_tokens])
        self.starts = np.concatenate([[0], np.cumsum(lens)])
        self.NP = int(self.starts[-1])
        self.P_tok = np.concatenate([np.array(k) for k in self.ex_tokens])
        self.P_i = np.concatenate([np.arange(1, n + 1) for n in lens])
        self.P_ex = np.repeat(np.arange(self.n_ex), lens)
        self.P_c = np.array([self.cls_id[p] for ps in pieces for p in ps])

        # ---- hash parameters (start from the brain's current values)
        self.d = np.array(brain.alphabet_directions, dtype=np.int64)
        self.s = np.array(brain.alphabet_start_positions, dtype=np.int64)

        # ---- per-token move lists
        occ_s, occ_d = {}, {}
        for e, toks in enumerate(self.ex_tokens):
            for i, t in enumerate(toks):
                occ_s.setdefault(t, []).append(self.starts[e] + i)
                for j, u in enumerate(toks):
                    if j != i:                      # token u at position j+1 shifts ball i by (j+1)*d[u]
                        dd = occ_d.setdefault(u, {})
                        dd[self.starts[e] + i] = dd.get(self.starts[e] + i, 0) + (j + 1)
        self.occ_s = {t: np.array(v) for t, v in occ_s.items()}
        self.occ_d = {t: (np.array(list(v)), np.array(list(v.values()))) for t, v in occ_d.items()}
        self.tokens = sorted(self.occ_s)
        self.vals_s = np.arange(self.D); self.vals_d = np.arange(-self.V, self.V + 1)
        self.flogf = np.zeros(self.NP + 2); k = np.arange(1, self.NP + 2); self.flogf[1:] = k * np.log(k)
        self._rebuild()

    # ---------------------------------------------------------------- classes / split
    def _reg(self, piece):
        if piece not in self.cls_id:
            self.cls_id[piece] = len(self.cls_str); self.cls_str.append(piece)

    def canonical_split(self, t, L):
        """Divide target t among L balls, right-aligned: leading balls stay silent, any surplus goes to the first balls."""
        if len(t) <= L:
            return [""] * (L - len(t)) + list(t)
        out, rest, balls_left = [], t, L
        surplus = len(t) - L
        for _ in range(L):
            take = 1 + min(surplus, self.max_len - 1)
            surplus -= take - 1
            out.append(rest[:take]); rest = rest[take:]
        return out

    # ---------------------------------------------------------------- hash + counts
    def addresses(self):
        dd = self.d[self.P_tok]; wd = dd * self.P_i
        W = np.add.reduceat(wd, self.starts[:-1])
        return (self.s[self.P_tok] + W[self.P_ex] - wd) % self.D

    def _rebuild(self):
        self.h = self.addresses()
        self.N = np.zeros((self.D, self.C), dtype=np.int32)
        np.add.at(self.N, (self.h, self.P_c), 1)
        self.nk = self.N.sum(1); self.mx = self.N.max(1)

    def conflicts(self):
        return int((self.nk - self.mx).sum())

    def ce(self):
        return float(self.flogf[self.nk].sum() - self.flogf[self.N].sum())

    def _remove(self, P):
        b = self.h[P]; np.subtract.at(self.N, (b, self.P_c[P]), 1)
        u = np.unique(b); self.nk[u] = self.N[u].sum(1); self.mx[u] = self.N[u].max(1)

    def _insert(self, P, hn):
        self.h[P] = hn; np.add.at(self.N, (hn, self.P_c[P]), 1)
        u = np.unique(hn); self.nk[u] = self.N[u].sum(1); self.mx[u] = self.N[u].max(1)

    def _local(self, buckets):
        u = np.unique(buckets)
        if self.loss == 'hard':
            return float((self.nk[u] - self.mx[u]).sum())
        return float(self.flogf[self.nk[u]].sum() - self.flogf[self.N[u]].sum())

    def _marginal(self, idx, c):
        """Exact change in the profiled loss from adding ONE position of class c to bucket idx (arrays)."""
        Nc = self.N[idx, c]
        if self.loss == 'hard':
            return (Nc < self.mx[idx]).astype(np.float64)
        n = self.nk[idx]
        return (self.flogf[n + 1] - self.flogf[n]) - (self.flogf[Nc + 1] - self.flogf[Nc])

    # ---------------------------------------------------------------- exact coordinate move
    def _move(self, P, w, cur, vals, ntry, slack=0.0):
        old = self.h[P].copy()
        before = self._local(old)
        self._remove(P)
        base = (old - w * cur) % self.D
        idx = (base[:, None] + w[:, None] * vals[None, :]) % self.D          # (m, n_candidates)
        cost = self._marginal(idx, self.P_c[P][:, None]).sum(0)
        cost = cost + self.rng.random(len(vals)) * (0.5 if self.loss == 'hard' else 1e-6)
        for vi in np.argsort(cost)[:ntry]:
            hn = (base + w * vals[vi]) % self.D
            self._insert(P, hn)
            if self._local(np.concatenate([old, hn])) <= before + slack + 1e-9:
                return vals[vi]
            self._remove(P)
        self._insert(P, old)
        return cur

    def sweep(self, tokens=None, ntry=3):
        toks = list(self.tokens if tokens is None else tokens)
        self.rng.shuffle(toks)
        for a in toks:
            Ps = self.occ_s[a]
            self.s[a] = self._move(Ps, np.ones(len(Ps), dtype=np.int64), self.s[a], self.vals_s, ntry)
            if a in self.occ_d:
                Pd, w = self.occ_d[a]
                self.d[a] = self._move(Pd, w, self.d[a], self.vals_d, ntry)

    def _kick(self, slack=3.0):
        """Escape a plateau: re-place one random token of a conflicting example, allowing a small temporary loss increase."""
        bad = np.where(self.N[self.h, self.P_c] < self.nk[self.h])[0]
        e = int(self.rng.choice(self.P_ex[bad])); a = int(self.rng.choice(self.ex_tokens[e]))
        Ps = self.occ_s[a]
        self.s[a] = self._move(Ps, np.ones(len(Ps), dtype=np.int64), self.s[a], self.vals_s, 40, slack)
        if a in self.occ_d:
            Pd, w = self.occ_d[a]
            self.d[a] = self._move(Pd, w, self.d[a], self.vals_d, 40, slack)

    def _repair(self, sweeps, kicks):
        """Focus on the tokens of examples that still conflict; kick when a sweep makes no progress; keep the best state."""
        best = (self.conflicts(), self.d.copy(), self.s.copy())
        for _ in range(sweeps):
            if self.conflicts() == 0: break
            before = self.conflicts()
            self.sweep(tokens=self._stuck_tokens(), ntry=40)
            if self.conflicts() >= before and kicks > 0:
                kicks -= 1; self._kick()
            if self.conflicts() < best[0]:
                best = (self.conflicts(), self.d.copy(), self.s.copy())
        if self.conflicts() > best[0]:
            self.d, self.s = best[1], best[2]; self._rebuild()

    def _stuck_tokens(self):
        bad = np.where(self.N[self.h, self.P_c] < self.nk[self.h])[0]      # every position in an impure bucket (incl. ties)
        return sorted({t for e in np.unique(self.P_ex[bad]) for t in self.ex_tokens[e]})

    # ---------------------------------------------------------------- latent split (hard EM)
    def table_strings(self):
        mode = self.N.argmax(1)
        return [self.cls_str[c] for c in mode]

    def resegment(self):
        """Re-divide each example's target among its balls so the pieces agree with what the table
        already emits. The loss cannot increase: every example's mismatches vs. the current table
        only go down, and the table is then re-optimised."""
        T = self.table_strings(); changed = 0
        for e in range(self.n_ex):
            a, b = self.starts[e], self.starts[e + 1]; L = b - a; t = self.ex_target[e]; n = len(t)
            tab = [T[self.h[p]] for p in range(a, b)]
            cur = [self.cls_str[c] for c in self.P_c[a:b]]
            if tab == cur:
                continue
            INF = 1e9
            dp = [[INF] * (n + 1) for _ in range(L + 1)]; bk = [[None] * (n + 1) for _ in range(L + 1)]
            dp[0][0] = 0.0
            for i in range(L):
                for o in range(n + 1):
                    if dp[i][o] >= INF: continue
                    for o2 in range(o, min(n, o + self.max_len) + 1):
                        piece = t[o:o2]
                        if piece not in self.cls_id: continue
                        c = dp[i][o] + (piece != tab[i]) + 1e-3 * (piece != cur[i])
                        if c < dp[i + 1][o2]:
                            dp[i + 1][o2] = c; bk[i + 1][o2] = (o, piece)
            o, new = n, []
            for i in range(L, 0, -1):
                o, piece = bk[i][o]; new.append(piece)
            new = new[::-1]
            if new != cur:
                self.P_c[a:b] = [self.cls_id[p] for p in new]; changed += 1
        self._rebuild()
        return changed

    # ---------------------------------------------------------------- metrics
    def table_exact_match(self):
        T = self.table_strings()
        return float(np.mean([''.join(T[self.h[p]] for p in range(self.starts[e], self.starts[e + 1])) == self.ex_target[e]
                              for e in range(self.n_ex)]))

    # ---------------------------------------------------------------- fit
    def fit(self, rounds=4, sweeps=8, repair_sweeps=40, kicks=15, verbose=True):
        t0 = time.time()
        if verbose: print(f"init: conflicts={self.conflicts()}/{self.NP}  ce={self.ce():.1f}  classes={self.C}")
        for r in range(rounds):
            prev, stall = None, 0
            for sw in range(sweeps):
                self.sweep()
                c = self.conflicts()
                if verbose: print(f"  round {r} sweep {sw}: conflicts={c}  ce={self.ce():.2f}  [{time.time()-t0:.1f}s]")
                if c == 0: break
                stall = stall + 1 if c == prev else 0; prev = c
                if stall >= 2: break
            self._repair(repair_sweeps, kicks)                  # focus on the examples that still conflict
            if verbose: print(f"  round {r} after repair: conflicts={self.conflicts()}")
            if self.conflicts() == 0 or not self.do_reseg: break
            ch = self.resegment()
            if verbose: print(f"  round {r} re-split {ch} examples: conflicts={self.conflicts()}")
        if verbose: print(f"done in {time.time()-t0:.1f}s: conflicts={self.conflicts()}/{self.NP}, table exact-match={self.table_exact_match():.4f}")
        return self

    # ---------------------------------------------------------------- compile to real deflectors
    def compile(self, verbose=True):
        """Write (d, s) and the table into the brain. A bucket whose string is 'abc' becomes the hole chain
        a -> b -> c -> sink on the nearest free deflectors. The ball energy is raised if a chain needs it."""
        br, D, BIG = self.brain, self.D, 10 ** 9
        br.alphabet_directions = [int(v) for v in self.d]
        br.alphabet_start_positions = [int(v) for v in self.s]
        T = self.table_strings()
        used = self.nk > 0
        free = ~used
        def nearest_free(x):
            for r in range(1, D):
                for y in (x - r, x + r):
                    if 0 <= y < D and free[y]:
                        return y
            raise RuntimeError("no free deflector left for a hole chain; increase D")
        sink = int(np.argmax(free)); free[sink] = False
        dl = [[k, -BIG] for k in range(D)]                       # default: silent bounce node
        need, max_hop, chain_nodes = 1, 0, 0
        for k in np.flatnonzero(used):
            S = T[k]
            if not S: continue
            nodes = [int(k)]
            for _ in S[1:]:
                u = nearest_free(nodes[-1]); free[u] = False; nodes.append(u); chain_nodes += 1
            for node, ch, nxt in zip(nodes, S, nodes[1:] + [sink]):
                dl[node] = [int(nxt), ch]
            cum = worst = 0
            for a, b in zip(nodes, nodes[1:]):
                cum += abs(b - a) - 1; worst = max(worst, cum); max_hop = max(max_hop, abs(b - a))
            need = max(need, worst + 1)
        br.deflectors_list = dl
        br.ball_energy = max(br.ball_energy, need)
        if verbose:
            print(f"compiled: {int(used.sum())} start buckets, {chain_nodes} extra chain nodes, "
                  f"longest emission={max(len(s) for s in T)} symbols, longest hop={max_hop}, ball_energy={br.ball_energy}")
        return br

## Train
`loss='ce'` profiles the cross-entropy; switch to `'hard'` for pure 0/1 conflicts. On the GPU-less default this is NumPy; every inner step is a gather/scatter-add, so it ports directly to PyTorch/CuPy and parallelises across tokens that share no example.

In [ ]:
brain = BallTrapBrain(alphabet, output, D=CFG['D'], bounce=CFG['bounce'], cap=CFG['cap'],
                      direction_range=CFG['direction_range'])
trainer = HashTrainer(brain, train_cases, loss=CFG['loss'], resegment=True, seed=CFG['seed'])
trainer.fit(rounds=10, sweeps=6, kicks=1)
brain = trainer.compile()

init: conflicts=1413/4708  ce=2863.6  classes=11
  round 0 sweep 0: conflicts=1236  ce=2480.88  [3.1s]
  round 0 sweep 1: conflicts=1236  ce=2480.88  [5.9s]
  round 0 sweep 2: conflicts=1191  ce=2432.28  [8.6s]
  round 0 sweep 3: conflicts=1221  ce=2396.15  [11.8s]
  round 0 sweep 4: conflicts=1221  ce=2396.15  [15.0s]
  round 0 sweep 5: conflicts=1221  ce=2396.15  [17.7s]
  round 0 after repair: conflicts=1221
  round 0 re-split 33 examples: conflicts=1185
  round 1 sweep 0: conflicts=1185  ce=2312.09  [144.4s]
  round 1 sweep 1: conflicts=1185  ce=2312.09  [147.3s]
  round 1 sweep 2: conflicts=1185  ce=2312.09  [150.0s]
  round 1 after repair: conflicts=1185
  round 1 re-split 0 examples: conflicts=1185
  round 2 sweep 0: conflicts=1185  ce=2312.09  [272.7s]
  round 2 sweep 1: conflicts=1185  ce=2312.09  [276.2s]
  round 2 sweep 2: conflicts=1185  ce=2312.09  [279.0s]


KeyboardInterrupt: 

## Evaluate on the real discrete simulator

In [ ]:
tr_exact, tr_ratio = evaluate(brain, train_cases)
te_exact, te_ratio = evaluate(brain, test_cases)
print(f"train: exact-match={tr_exact:.4f}  similarity={tr_ratio:.4f}")
print(f"test : exact-match={te_exact:.4f}  similarity={te_ratio:.4f}")
for inp, ans in train_cases[:6]:
    print(f"{inp!r:34} -> {brain.compute_input(inp)!r:10} (target {ans!r})")

### Sanity check: the decomposition `output = concat_i T[h_i]` holds on the real simulator

In [ ]:
T = brain.build_table()
ok = all(brain.compute_input(inp) == ''.join(T[k] for k in brain.start_nodes(inp)) for inp, _ in cases)
print("compute_input == concat of table lookups on all cases:", ok)
multi = sorted({s for s in T if len(s) > 1}, key=len, reverse=True)[:5]
print("examples of multi-symbol buckets (one ball emitting several symbols):", multi)

## Notes and limits

* **What this is.** A learned perfect-hash / associative memory: capacity is `D` buckets, and fitting it is exact and cheap. It memorises the training set; held-out accuracy only appears if the task has structure the position-weighted hash can express, so don't expect it from a single paragraph of text.
* **Why there is no gradient and no GA.** The only hard part of the original problem is the hash `(d, s)`; the table and the split are solved exactly, and each hash parameter is searched exhaustively over its whole range, so nothing has to be annealed or relaxed.
* **Scaling knobs.** `loss='ce'` vs `'hard'`; `D` (capacity); `cap` (max symbols per ball). For large `D·C`, store `N` sparsely (a dict keyed by bucket) — only the buckets actually hit are ever touched.
* **Depth.** Because the walk cannot read the input, the model has a single lookup. Layering (letting the walk consume tokens, so a hop's address is a hash of the previous node and the next token) would reuse the same exact search per layer, but is a change to the architecture rather than to the trainer.